In [1]:
import glob
import math
import os
import pickle
import queue
import re
import threading
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from itertools import islice
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

Model class - dihedral

In [2]:
class AE(nn.Module):
    def __init__(self, n_in, latent, HIDDEN):
        super().__init__()
        h1, h2 = HIDDEN
        self.enc = nn.Sequential(nn.Linear(n_in, h1), nn.GELU(), nn.Linear(h1, h2), nn.GELU(),
                                 nn.Linear(h2, latent))
        self.dec = nn.Sequential(nn.Linear(latent, h2), nn.GELU(), nn.Linear(h2, h1), nn.GELU(),
                                 nn.Linear(h1, n_in), nn.Tanh())

    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

Funkcje pomocnicze - dihedral

In [3]:
replacement_map = {
    "Res1_Beta": "b1", "Res1_Gamma": "g1", "Res1_Delta": "d1", "Res1_Epsilon": "e1", "Res1_Zeta": "z1",
    "Res2_Alpha": "a2", "Res2_Beta": "b2", "Res2_Gamma": "g2", "Res2_Delta": "d2", "Res2_Epsilon": "e2", "Res2_Zeta": "z2",
    "Res3_Alpha": "a3", "Res3_Beta": "b3", "Res3_Gamma": "g3", "Res3_Delta": "d3", "Res3_Epsilon": "e3", "Res3_Zeta": "z3",
    "Res4_Alpha": "a4", "Res4_Beta": "b4", "Res4_Gamma": "g4", "Res4_Delta": "d4", "Res4_Epsilon": "e4", "Res4_Zeta": "z4",
    "Res5_Alpha": "a5", "Res5_Beta": "b5", "Res5_Gamma": "g5", "Res5_Delta": "d5", "Res5_Epsilon": "e5", "Res5_Zeta": "z5",
    "Res6_Alpha": "a6", "Res6_Beta": "b6", "Res6_Gamma": "g6", "Res6_Delta": "d6", "Res6_Epsilon": "e6", "Res6_Zeta": "z6",
    "Res7_Alpha": "a7", "Res7_Beta": "b7", "Res7_Gamma": "g7", "Res7_Delta": "d7", "Res7_Epsilon": "e7", "Res7_Zeta": "z7",
    "Res8_Alpha": "a8", "Res8_Beta": "b8", "Res8_Gamma": "g8", "Res8_Delta": "d8", "Res8_Epsilon": "e8", "Res8_Zeta": "z8",
    "Res9_Alpha": "a9", "Res9_Beta": "b9", "Res9_Gamma": "g9", "Res9_Delta": "d9", "Res9_Epsilon": "e9", "Res9_Zeta": "z9",
    "Res10_Alpha": "a10", "Res10_Beta": "b10", "Res10_Gamma": "g10", "Res10_Delta": "d10", "Res10_Epsilon": "e10", "Res10_Zeta": "z10",
    "Res11_Alpha": "a11", "Res11_Beta": "b11", "Res11_Gamma": "g11", "Res11_Delta": "d11", "Res11_Epsilon": "e11", "Res11_Zeta": "z11",
    "Res12_Alpha": "a12", "Res12_Beta": "b12", "Res12_Gamma": "g12", "Res12_Delta": "d12", "Res12_Epsilon": "e12", "Res12_Zeta": "z12",
    "Res13_Alpha": "a13", "Res13_Beta": "b13", "Res13_Gamma": "g13", "Res13_Delta": "d13", "Res13_Epsilon": "e13", "Res13_Zeta": "z13",
    "Res14_Alpha": "a14", "Res14_Beta": "b14", "Res14_Gamma": "g14", "Res14_Delta": "d14", "Res14_Epsilon": "e14", "Res14_Zeta": "z14",
    "Res15_Alpha": "a15", "Res15_Beta": "b15", "Res15_Gamma": "g15", "Res15_Delta": "d15", "Res15_Epsilon": "e15", "Res15_Zeta": "z15",
    "Res16_Alpha": "a16", "Res16_Beta": "b16", "Res16_Gamma": "g16", "Res16_Delta": "d16", "Res16_Epsilon": "e16", "Res16_Zeta": "z16",
    "Res17_Alpha": "a17", "Res17_Beta": "b17", "Res17_Gamma": "g17", "Res17_Delta": "d17", "Res17_Epsilon": "e17", "Res17_Zeta": "z17",
    "Res18_Alpha": "a18", "Res18_Beta": "b18", "Res18_Gamma": "g18", "Res18_Delta": "d18", "Res18_Epsilon": "e18", "Res18_Zeta": "z18",
    "Res19_Alpha": "a19", "Res19_Beta": "b19", "Res19_Gamma": "g19", "Res19_Delta": "d19", "Res19_Epsilon": "e19", "Res19_Zeta": "z19",
    "Res20_Alpha": "a20", "Res20_Beta": "b20", "Res20_Gamma": "g20", "Res20_Delta": "d20", "Res20_Epsilon": "e20"
}

def load_model(cpkt, device='cpu'):
    ck=torch.load(cpkt, map_location=device, weights_only=False)
    model = AE(ck['n_in'], ck['latent'], HIDDEN=tuple(ck['hidden']))
    model.load_state_dict(ck['state_dict'])
    model.eval().to(device)
    return model, list(ck['all_names']), ck


def read_csv_run(path, names=None, stride=1, sep=None, on_nan="raise"):

    df = pd.read_csv(path, sep=sep, engine="python")
    df = df.drop(columns=['File'],)
    df = df.rename(columns = replacement_map)
    names = list(df.columns)
    names = list(names)
    deg = df.to_numpy(dtype=np.float32)
    return deg, names

def prepare_input(deg, names, all_names):
    pos = {n: i for i, n in enumerate(all_names)}
    keep = [j for j, n in enumerate(names) if n in pos]
    unknown = [n for n in names if n not in pos]
    if unknown:
        warnings.warn(f"{len(unknown)} column(s) are not known to the model and are dropped: {unknown[:8]}")
    if not keep:
        raise ValueError("none of the columns match the model's all_names")
    idx = [pos[names[j]] for j in keep]
    D = len(all_names)
    full = np.zeros((len(deg), D), np.float32)
    full[:, idx] = np.asarray(deg, np.float32)[:, keep]
    pres = np.zeros(D, bool)
    pres[idx] = True
    m = pres.astype(np.float32)
    r = np.deg2rad(full).astype(np.float32)
    x = np.concatenate([np.cos(r) * m, np.sin(r) * m], axis=1).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0).astype(np.float32)
    mask = np.concatenate([m, m])
    return x, mask, full, pres

@torch.no_grad()
def reconstruct(model, x, mask, device="cpu", batch=16384):
    """-> (reconstruction (N, 2D) with padded entries forced to 0, latent codes (N, latent))."""
    recs, zs = [], []
    for i in range(0, len(x), batch):
        r, z = model(torch.from_numpy(np.ascontiguousarray(x[i:i + batch])).to(device))
        recs.append(r.cpu().numpy())
        zs.append(z.cpu().numpy())
    return np.concatenate(recs) * mask, np.concatenate(zs)


def to_degrees(rec):
    """(N, 2D) [cos | sin] -> (N, D) degrees in (-180, 180]. Only the direction of each (cos, sin) pair matters."""
    D = rec.shape[1] // 2
    return np.degrees(np.arctan2(rec[:, D:], rec[:, :D]))


def angular_error(true_deg, pred_deg):
    """Wrap-safe absolute difference in degrees (179 vs -179 -> 2)."""
    return np.abs((pred_deg - true_deg + 180.0) % 360.0 - 180.0)


def evaluate_array(model, all_names, deg, names, device="cpu"):
    x, mask, full, valid = prepare_input(deg, names, all_names)
    rec, z = reconstruct(model, x, mask, device)      # output already zeroed where masked
    pred = to_degrees(rec)
    pred = np.where(valid, pred, np.nan)              # no prediction where the angle does not exist
    true = np.where(valid, full, np.nan)              # same for the "true" side
    err = angular_error(true, pred)                   # NaN wherever either side is missing
    return dict(x=x, mask=mask, true_deg=true, pred_deg=pred, err=err, latent=z,
                present=valid.any(0))


def evaluate_file(model, all_names, path, stride=1, device="cpu"):
    deg, names = read_run(path, stride)
    return evaluate_array(model, all_names, deg, names, device)


def evaluate_csv(model, all_names, path, device="cpu", **read_kwargs):
    """read_kwargs go to read_csv_run (names, ref_txt, time_col, stride, sep, on_nan)."""
    deg, names = read_csv_run(path, **read_kwargs)
    return evaluate_array(model, all_names, deg, names, device)


main - dihedral

In [4]:
if __name__ == '__main__':
    checkpoint = 'AE_dihedral_261026/dihedral_ae_n_v1/all_runs_streamed/models/ae_latent64.pt'
    data_input = 'AE_test.csv'
    dev = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
    model, all_names, ck = load_model(checkpoint, dev)
    if data_input.endswith(".csv"):
        out = evaluate_csv(model, all_names, data_input, dev)
    else:
        out = evaluate_file(model, all_names, data_input, stride=int(ck.get("stride", 1)), device=dev)
    e = out["err"]
    ids = pd.read_csv(data_input, sep=None, engine="python")["File"].astype(str).to_numpy()
    err, names_arr = out["err"], np.array(all_names)
    valid = ~np.isnan(err)

    rows = []
    for i in range(len(err)):
        v = valid[i]
        if not v.any():
            rows.append(dict(structure=ids[i], n_dihedrals=0)); continue
        worst = np.argsort(np.nan_to_num(err[i], nan=-1))[::-1][:3]
        rows.append(dict(structure=ids[i], n_dihedrals=int(v.sum()),
                         mean_err=err[i][v].mean(), median_err=np.median(err[i][v]),
                         p95_err=np.percentile(err[i][v], 95),
                         p99_err=np.percentile(err[i][v], 99), max_err=err[i][v].max(),
                         worst_3=", ".join(names_arr[worst])))
    per_frame = pd.DataFrame(rows)
    print(per_frame.round(2).to_string(index=False))
    print("mean over structures (each counts equally):", round(per_frame.mean_err.mean(), 2),
          "| pooled over all values:", round(float(np.nanmean(err)), 2))




  structure  n_dihedrals  mean_err  median_err   p95_err    p99_err    max_err       worst_3
 file_1.pdb           81 13.280000        8.32 42.919998  50.500000  53.450001   b12, a9, b9
 file_2.pdb           81 14.280000        8.80 45.540001  54.919998  57.820000   b12, a9, b9
 file_3.pdb           81 12.010000        7.24 37.709999  50.169998  51.639999  b4, b12, b13
 file_4.pdb           81 14.210000       10.52 45.500000  51.610001  56.060001   a9, b9, z12
 file_5.pdb           81 13.770000        9.68 39.090000  58.750000  61.740002   g1, b12, e7
 file_6.pdb           93 13.750000        8.81 41.080002  52.169998  70.639999    a9, e1, a8
 file_7.pdb           93  9.780000        6.28 28.930000  41.869999  64.500000  g1, g15, a13
 file_8.pdb           93  9.570000        5.39 28.250000  54.900002 120.910004   g1, a13, z3
 file_9.pdb           93 10.110000        6.15 30.129999  72.040001  77.269997    b9, g9, a9
file_10.pdb           93 13.240000        7.22 32.970001  96.269997 15